# Chroma Cloud: RAG from ingestion to retrieval

In `01_chromadb.ipynb` we stored the chunks in a **local** Chroma folder. Here we build the same RAG pipeline on **Chroma Cloud**, Chroma's managed service, so the data lives online and any notebook or app with the API key can use it.

Every step is a small **helper function**, so you can reuse them in other notebooks or move them to a `.py` file later:

```
INGESTION  (run once, and again when the files change)

 data/raw/text/*.txt ─► load_documents() ─► add_topic_metadata() ─► split_documents() ─► make_chunk_ids()
                                                                                                │
                                                                                                ▼
                                                    ingest_chunks(): embed + upsert in batches ─► Chroma Cloud collection

RETRIEVAL + GENERATION  (every question)

 question ─► get_retriever() ─► top-k chunks ─► build_rag_chain(): prompt + LLM ─► ask(): answer + sources
```

| Step | Helper | What it does |
|---|---|---|
| Connect | `get_chroma_cloud_client()` | Connects to Chroma Cloud with the API key from `.env` |
| | `get_cloud_vectorstore()` | Opens (or creates) a collection as a LangChain vector store |
| | `show_collections()` | Lists the collections in the cloud database and their sizes |
| Ingest | `load_documents()` | Loads every `.txt` file in a folder |
| | `add_topic_metadata()` | Adds a `topic` to each document, for filtering later |
| | `split_documents()` | Splits the documents into chunks |
| | `make_chunk_ids()` | Stable ids per file: `ai_agents_chunk_0`, `ai_agents_chunk_1`, ... |
| | `ingest_chunks()` | Embeds and upserts the chunks in batches |
| | `run_ingestion()` | Runs all the ingestion steps in one call |
| Retrieve | `get_retriever()` | A retriever with `k` and an optional metadata filter |
| | `search_with_scores()` | Prints the closest chunks with their distances |
| Generate | `format_docs()` | Joins the chunks into the prompt context, with `[Source: ...]` labels |
| | `build_rag_chain()` | An LCEL chain that returns the answer **and** the chunks it used |
| | `ask()` | Runs the chain with `ainvoke` and prints the answer and its sources |

### Before you start

Add your Chroma Cloud key to the `.env` file in the project root:

```
CHROMA_API_KEY=...
CHROMA_DATABASE=...     # the database name from the Chroma Cloud dashboard
CHROMA_TENANT=...       # optional: inferred from the API key
```

`CHROMA_DATABASE` can only be left out when the API key is scoped to **one** database. A key created for the whole tenant needs it.

`OPENAI_API_KEY` is used for the embeddings and the LLM, as in the other notebooks.

In [1]:
## Setup: shared project paths from src.config, which also loads the API keys from .env
import os
from pathlib import Path
from dotenv import load_dotenv

from src.config import PROJECT_ROOT, RAW_TEXT_DIR

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp


In [2]:
## Load libraries
import warnings
warnings.filterwarnings("ignore")

from collections import defaultdict

import chromadb
from chromadb.errors import ChromaAuthError

# LangChain core imports
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# LangChain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.vectorstores import Chroma

In [3]:
## Settings used by the helpers
DATA_DIR = RAW_TEXT_DIR
COLLECTION_NAME = "rag_bootcamp_docs"   # the Chroma Cloud collection used by this notebook

EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-5"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
TOP_K = 3

TOPIC_BY_FILE = {
    "ai_agents.txt": "agents",
    "machine_learning_algorithms.txt": "ml",
    "reinforcement_learning.txt": "rl",
    "deep_learning.txt": "dl",
}

## 1. Connect to Chroma Cloud

In [4]:
## Helpers: connect to Chroma Cloud and open a collection as a LangChain vector store
def get_chroma_cloud_client():
    """Connect to Chroma Cloud with CHROMA_API_KEY, CHROMA_DATABASE (and optionally CHROMA_TENANT) from .env."""
    load_dotenv()   # picks up keys added to .env after the kernel started
    if not os.getenv("CHROMA_API_KEY"):
        raise ValueError("CHROMA_API_KEY is missing: add it to the .env file in the project root and save the file")

    try:
        client = chromadb.CloudClient(
            api_key=os.getenv("CHROMA_API_KEY"),
            tenant=os.getenv("CHROMA_TENANT"),       # None: inferred from the API key
            database=os.getenv("CHROMA_DATABASE"),   # None: inferred only if the key is scoped to one database
        )
    except ChromaAuthError as error:
        raise ValueError(
            "Chroma Cloud can't tell which database to use from this API key. "
            "Add CHROMA_DATABASE=<database name from the Chroma Cloud dashboard> to .env"
        ) from error

    identity = client.get_user_identity()   # an authenticated call: fails here if the key is wrong
    print(f"Connected to Chroma Cloud | tenant: {identity.tenant} | databases: {identity.databases}")
    return client


def get_cloud_vectorstore(client, embedding, collection_name=COLLECTION_NAME):
    """Open the collection in Chroma Cloud (it is created if it doesn't exist) as a LangChain vector store."""
    return Chroma(
        client=client,
        collection_name=collection_name,
        embedding_function=embedding,
    )


def show_collections(client):
    """Print every collection in the Chroma Cloud database with its number of chunks."""
    collections = client.list_collections()
    print(f"{len(collections)} collection(s) in Chroma Cloud:")
    for collection in collections:
        print(f"  - {collection.name}: {collection.count()} chunks")

## 2. Data ingestion

In [5]:
## Helpers: load, tag, split, give each chunk a stable id, and upsert into Chroma Cloud
def load_documents(doc_dir=DATA_DIR, glob="*.txt"):
    """Load every matching file in doc_dir as a LangChain Document."""
    loader = DirectoryLoader(doc_dir, glob=glob, loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
    documents = loader.load()
    print(f"Loaded {len(documents)} documents from {doc_dir}")
    return documents


def add_topic_metadata(documents, topic_by_file=TOPIC_BY_FILE):
    """Add a "topic" to each document's metadata, based on its file name ("general" if the file isn't listed)."""
    for doc in documents:
        doc.metadata["topic"] = topic_by_file.get(Path(doc.metadata["source"]).name, "general")
    return documents


def split_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """Split the documents into chunks. Each chunk keeps the metadata of its document."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],   # try paragraphs first, then lines, then words, then characters
    )
    chunks = text_splitter.split_documents(documents)
    print(f"Split into {len(chunks)} chunks")
    return chunks


def make_chunk_ids(chunks):
    """Stable ids per file (ai_agents_chunk_0, ai_agents_chunk_1, ...), so re-running upserts instead of duplicating."""
    position = defaultdict(int)
    ids = []
    for chunk in chunks:
        stem = Path(chunk.metadata["source"]).stem
        ids.append(f"{stem}_chunk_{position[stem]}")
        position[stem] += 1
    return ids


def ingest_chunks(vectorstore, chunks, ids, batch_size=100):
    """Embed the chunks and upsert them into the collection, batch_size chunks per request."""
    for start in range(0, len(chunks), batch_size):
        end = min(start + batch_size, len(chunks))
        vectorstore.add_documents(documents=chunks[start:end], ids=ids[start:end])
        print(f"  upserted chunks {start + 1} to {end}")
    print(f"Collection '{vectorstore._collection.name}' now has {vectorstore._collection.count()} chunks")


def run_ingestion(vectorstore, doc_dir=DATA_DIR):
    """Load, tag, split and upsert every text file in doc_dir. Returns the chunks."""
    documents = add_topic_metadata(load_documents(doc_dir))
    chunks = split_documents(documents)
    ingest_chunks(vectorstore, chunks, make_chunk_ids(chunks))
    return chunks

## 3. Retrieval

Chroma returns **distances** (squared L2 by default): **lower = more similar**. Metadata filters use Chroma's `where` syntax, such as `{"topic": "rl"}` or `{"topic": {"$in": ["ml", "dl"]}}`, and run **inside the database**, so you still get `k` results when at least `k` chunks match.

In [6]:
## Helpers: a retriever with an optional metadata filter, and a scored search for checking results
def get_retriever(vectorstore, k=TOP_K, filter=None):
    """A retriever that returns the k closest chunks, optionally only the ones that match a metadata filter."""
    search_kwargs = {"k": k}
    if filter:
        search_kwargs["filter"] = filter
    return vectorstore.as_retriever(search_kwargs=search_kwargs)


def search_with_scores(vectorstore, query, k=TOP_K, filter=None):
    """Print the k closest chunks with their distance (lower = more similar)."""
    results = vectorstore.similarity_search_with_score(query, k=k, filter=filter)

    print(f"Query : {query}")
    print(f"Filter: {filter}\n")
    for i, (doc, score) in enumerate(results, start=1):
        print(f"--- Result {i} | distance: {score:.4f} | topic: {doc.metadata.get('topic')} | {doc.metadata.get('source')} ---")
        print(doc.page_content[:200] + "...\n")
    return results

## 4. RAG chain

In [7]:
## Helpers: format the context, build the LCEL RAG chain, and ask questions (async)
rag_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question.
If you don't know the answer based on the context, say you don't know.
Mention the source file(s) you used.

Context:
{context}

Question: {question}

Answer:""")


def format_docs(docs):
    """Join the retrieved chunks into one string, each labelled with its source file."""
    return "\n\n".join(f"[Source: {doc.metadata['source']}]\n{doc.page_content}" for doc in docs)


def build_rag_chain(retriever, llm, prompt=rag_prompt):
    """LCEL chain: question -> {"docs": [Document, ...], "question": "...", "answer": "..."}."""
    answer_chain = (
        RunnablePassthrough.assign(context=lambda x: format_docs(x["docs"]))   # Documents -> context string
        | prompt
        | llm
        | StrOutputParser()
    )
    return (
        {"docs": retriever, "question": RunnablePassthrough()}
        | RunnablePassthrough.assign(answer=answer_chain)   # adds "answer", keeps "docs" and "question"
    )


async def ask(chain, question):
    """Run the chain with ainvoke, then print the answer and the metadata of the chunks it used."""
    result = await chain.ainvoke(question)

    print(f"Question: {question}\n")
    print(f"Answer:\n{result['answer']}\n")
    print(f"Sources ({len(result['docs'])} chunks):")
    for doc in result["docs"]:
        print(f"  - topic: {doc.metadata.get('topic')} | {doc.metadata.get('source')}")
    return result

## 5. Run the pipeline

In [8]:
## Connect to Chroma Cloud and open the collection
cloud_client = get_chroma_cloud_client()

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vectorstore = get_cloud_vectorstore(cloud_client, embeddings)

show_collections(cloud_client)

Connected to Chroma Cloud | tenant: 64279673-009f-4696-bb1e-3ab4ca1b6960 | databases: []


/var/folders/j5/52bx342j7k59wyrt9n978pp00000gn/T/ipykernel_23838/1100955973.py:27: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  return Chroma(


1 collection(s) in Chroma Cloud:
  - rag_bootcamp_docs: 0 chunks


In [9]:
## Ingest: load, tag, split and upsert every text file (only needed once, or when the files change)
chunks = run_ingestion(vectorstore)

print(f"\nFirst chunk metadata: {chunks[0].metadata}")

Loaded 4 documents from ./data/raw/text
Split into 44 chunks
  upserted chunks 1 to 44
Collection 'rag_bootcamp_docs' now has 44 chunks

First chunk metadata: {'source': 'data/raw/text/ai_agents.txt', 'topic': 'agents'}


In [10]:
## Retrieval: check what the vector store returns before adding the LLM
results = search_with_scores(vectorstore, "What is reinforcement learning?")

Query : What is reinforcement learning?
Filter: None

--- Result 1 | distance: 0.6486 | topic: rl | data/raw/text/reinforcement_learning.txt ---
Reinforcement Learning: An Introduction

Reinforcement learning (RL) is a branch of machine learning in which an agent learns to make decisions by interacting with an environment. Instead of learning ...

--- Result 2 | distance: 0.6863 | topic: ml | data/raw/text/machine_learning_algorithms.txt ---
Reinforcement Learning
In reinforcement learning, an agent learns by interacting with an environment. It takes actions, receives rewards or penalties, and learns a strategy that maximizes total reward...

--- Result 3 | distance: 0.7468 | topic: rl | data/raw/text/reinforcement_learning.txt ---
Conclusion
Reinforcement learning teaches agents to make a sequence of good decisions through trial and error. Combined with deep learning, it has produced systems that beat human champions at complex...



In [11]:
## Retrieval with a metadata filter: only chunks with topic = ml
results = search_with_scores(vectorstore, "What is reinforcement learning?", filter={"topic": "ml"})

Query : What is reinforcement learning?
Filter: {'topic': 'ml'}

--- Result 1 | distance: 0.6863 | topic: ml | data/raw/text/machine_learning_algorithms.txt ---
Reinforcement Learning
In reinforcement learning, an agent learns by interacting with an environment. It takes actions, receives rewards or penalties, and learns a strategy that maximizes total reward...

--- Result 2 | distance: 1.3022 | topic: ml | data/raw/text/machine_learning_algorithms.txt ---
Supervised Learning
In supervised learning, the model is trained on labeled data, meaning each input comes with the correct output. The goal is to predict the output for new, unseen inputs. Supervised...

--- Result 3 | distance: 1.3406 | topic: ml | data/raw/text/machine_learning_algorithms.txt ---
Conclusion
Machine learning algorithms are the foundation of modern AI systems. Understanding their strengths and weaknesses helps you choose the right tool for each problem, and more advanced techniq...



In [12]:
## RAG chain: retriever + prompt + LLM
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)
rag_chain = build_rag_chain(get_retriever(vectorstore), llm)

result = await ask(rag_chain, "What are the types of neural networks?")

Question: What are the types of neural networks?

Answer:
- Convolutional Neural Networks (CNNs) — designed for images
- Recurrent Neural Networks (RNNs) — for sequential data
- Long Short-Term Memory networks (LSTMs) — a type of RNN for longer sequences

Source: data/raw/text/deep_learning.txt

Sources (3 chunks):
  - topic: dl | data/raw/text/deep_learning.txt
  - topic: dl | data/raw/text/deep_learning.txt
  - topic: dl | data/raw/text/deep_learning.txt


In [13]:
result = await ask(rag_chain, "What are the core components of an AI agent?")

Question: What are the core components of an AI agent?

Answer:
- Large Language Model (LLM) for reasoning and decision-making (the agent’s “brain”)
- Tools (e.g., web search, calculator) for taking actions (the agent’s “hands”)
- Planning to execute multi-step tasks
- Memory to keep track of context and intermediate results
- Retrieval (e.g., RAG) to access accurate, up-to-date knowledge

Source: data/raw/text/ai_agents.txt

Sources (3 chunks):
  - topic: agents | data/raw/text/ai_agents.txt
  - topic: agents | data/raw/text/ai_agents.txt
  - topic: agents | data/raw/text/ai_agents.txt


In [14]:
## A RAG chain that only answers from deep learning chunks (topic = dl)
dl_rag_chain = build_rag_chain(get_retriever(vectorstore, filter={"topic": "dl"}), llm)

# deep_learning.txt doesn't cover reinforcement learning, so the LLM should say it doesn't know
result = await ask(dl_rag_chain, "What is reinforcement learning?")

Question: What is reinforcement learning?

Answer:
I don’t know based on the provided context; reinforcement learning isn’t defined there.

Source: data/raw/text/deep_learning.txt

Sources (3 chunks):
  - topic: dl | data/raw/text/deep_learning.txt
  - topic: dl | data/raw/text/deep_learning.txt
  - topic: dl | data/raw/text/deep_learning.txt


## Reconnecting later

The chunks are stored in Chroma Cloud, so another notebook or app can skip the ingestion step and go straight to retrieval:

```python
cloud_client = get_chroma_cloud_client()
vectorstore = get_cloud_vectorstore(cloud_client, OpenAIEmbeddings(model="text-embedding-3-small"))
rag_chain = build_rag_chain(get_retriever(vectorstore), ChatOpenAI(model="gpt-5", temperature=0))

result = await ask(rag_chain, "What is deep learning?")
```

- **Same embedding model.** Always query with the model that embedded the chunks (`text-embedding-3-small` here).
- **Adding files.** Put new `.txt` files in `data/raw/text` and run `run_ingestion(vectorstore)` again. The stable ids mean existing chunks are overwritten, not duplicated.
- **Starting over.** `cloud_client.delete_collection(COLLECTION_NAME)` deletes the collection and its data in the cloud, for every app that uses it.

In [15]:
cloud_client = get_chroma_cloud_client()
vectorstore = get_cloud_vectorstore(cloud_client, OpenAIEmbeddings(model="text-embedding-3-small"))
rag_chain = build_rag_chain(get_retriever(vectorstore), ChatOpenAI(model="gpt-5", temperature=0))

result = await ask(rag_chain, "What is deep learning?")

Connected to Chroma Cloud | tenant: 64279673-009f-4696-bb1e-3ab4ca1b6960 | databases: []
Question: What is deep learning?

Answer:
Deep learning is a subfield of machine learning that uses artificial neural networks with many layers to learn patterns from data; “deep” refers to the number of layers between input and output, with each layer learning increasingly abstract representations.

Source: data/raw/text/deep_learning.txt

Sources (3 chunks):
  - topic: dl | data/raw/text/deep_learning.txt
  - topic: dl | data/raw/text/deep_learning.txt
  - topic: dl | data/raw/text/deep_learning.txt
